<a href="https://colab.research.google.com/github/OscarPellicer/extra-attributions/blob/main/notebooks/01_tabular_xai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 1 · Hands-on: Explainability for tabular models

We explain an **XGBoost** model that predicts house prices (California housing) with the methods of the first session: **permutation importance**, **PDP/ICE**, **SHAP** and **counterfactuals** (DiCE).

Run the cells in order. Parameters marked `# change me` list values worth trying: change one, re-run the cell and see what happens. You never need to write code.

*A longer version with more methods, exercises and answers is in `extended/01_tabular_xai.ipynb`.*

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    %pip install -q shap xgboost lime dice-ml
else:
    print("Running locally: if any import below fails, run `pip install -r requirements.txt`")

In [ ]:
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import sklearn
import xgboost
import shap
import lime
import lime.lime_tabular
import dice_ml

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.neighbors import NearestNeighbors

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Plot style and some cosmetic settings
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)
warnings.filterwarnings('ignore', category=PendingDeprecationWarning)  # matplotlib deprecations triggered inside shap
warnings.filterwarnings('ignore', category=FutureWarning)

print(f"sklearn {sklearn.__version__} | xgboost {xgboost.__version__} | shap {shap.__version__}")

## Data and model

**California housing** (1990 census): each row is a district; the target is the **median house value** in units of **$100k** (capped at 5.0). Features: median income (`MedInc`), house age, average rooms / bedrooms / occupants per household, population, and `Latitude` / `Longitude`.

In [ ]:
# Downloads the dataset automatically (cached after the first run)
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
feature_names = list(X.columns)

print("Shape:", X.shape)
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

model = xgboost.XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=SEED, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"XGBoost test R2: {r2_score(y_test, y_pred):.3f}")

## 1. Global feature importance

**Built-in importance** comes from the trees themselves (computed on *training* data): `gain` = loss reduction of the splits on a feature, `weight` = number of splits, `cover` = samples affected.

**Permutation importance (PFI)** shuffles one feature at a time on held-out data and measures the drop in R². It works for any model, but shuffling also creates unrealistic districts (e.g. the latitude of San Francisco with the longitude of Los Angeles).

In [ ]:
IMPORTANCE_TYPE = 'gain'  # change me: 'weight', 'gain' or 'cover'

booster_imp = pd.Series(model.get_booster().get_score(importance_type=IMPORTANCE_TYPE))
booster_imp = booster_imp.reindex(feature_names).fillna(0)
booster_imp = booster_imp / booster_imp.sum()  # normalize to sum 1

ax = booster_imp.sort_values().plot.barh(figsize=(8, 4), color='tab:gray')
ax.set_xlabel(f'Normalized importance ({IMPORTANCE_TYPE})')
ax.set_title(f'XGBoost built-in importance: {IMPORTANCE_TYPE}')
plt.show()

In [ ]:
PERM_ON = 'test'  # change me: 'test' or 'train'

X_perm, y_perm = (X_test, y_test) if PERM_ON == 'test' else (X_train, y_train)
perm = permutation_importance(model, X_perm, y_perm, n_repeats=10, random_state=SEED, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=feature_names)

perm_std = pd.Series(perm.importances_std, index=feature_names)
order = perm_imp.sort_values().index
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(order, perm_imp[order], xerr=perm_std[order], color='tab:green', capsize=3)
ax.set_xlabel('Decrease in R² when the feature is permuted')
ax.set_title(f'Permutation importance (on {PERM_ON} set)')
plt.show()

## 2. PDP and ICE

The **PDP** of a feature is the average prediction when we set that feature to a value $v$ for **every** district: $\mathrm{PD}(v) = rac{1}{n}\sum_i f(v, x_C^{(i)})$. It shows *how* the prediction changes with the feature (the ticks at the bottom are the deciles of the data).

In [ ]:
X_pdp = X_train.sample(1000, random_state=SEED)  # subsample to keep PDP/ICE fast

features_to_plot = ['MedInc', 'AveOccup', 'HouseAge', 'Latitude', 'Longitude', 'AveRooms']

fig, ax = plt.subplots(figsize=(14, 7))
PartialDependenceDisplay.from_estimator(model, X_pdp, features_to_plot, kind='average', n_cols=3, ax=ax)
plt.suptitle('Partial Dependence Plots (PDP)', y=1.02)
plt.tight_layout()
plt.show()

**2D PDP**: how two features *jointly* affect the prediction (interactions).

In [ ]:
features_2d = [
    ('Longitude', 'Latitude'),  # geographic interaction
    ('MedInc', 'AveOccup')      # socioeconomic interaction
]

fig, ax = plt.subplots(figsize=(14, 6))
PartialDependenceDisplay.from_estimator(model, X_pdp, features_2d, kind='average',
                                        grid_resolution=25, n_cols=2, ax=ax)
plt.suptitle('2D Partial Dependence Plots (interactions)', y=1.02)
plt.tight_layout()
plt.show()

**ICE**: one curve per district (the PDP is their average). Non-parallel curves mean **interactions** that the PDP hides (`HouseAge` is a good example: its PDP is almost flat, but individual curves go up *and* down).

In [ ]:
ICE_FEATURE = 'MedInc'  # change me: 'MedInc', 'AveOccup', 'HouseAge', 'Latitude', ...
CENTERED = False        # change me: True for centered ICE (c-ICE)

fig, ax = plt.subplots(figsize=(8, 6))
PartialDependenceDisplay.from_estimator(
    model, X_pdp, [ICE_FEATURE],
    kind='both',       # individual ICE lines + their average (PDP)
    subsample=100,     # draw only 100 ICE curves to avoid clutter
    centered=CENTERED, random_state=SEED, ax=ax)
ax.set_title(f'ICE plot for {ICE_FEATURE} (centered={CENTERED})')
plt.show()

### The correlated-features pitfall

To compute the PDP of `Latitude`, every district gets every latitude value **while keeping its own longitude**: many of these points are in the ocean or the desert, where the model has never seen data (this is what ALE avoids).

In [ ]:
PITFALL_PAIR = ('Longitude', 'Latitude')  # change me: ('AveRooms', 'AveBedrms'); the PDP varies the 2nd feature

a, b = PITFALL_PAIR
rows = X_pdp.sample(200, random_state=SEED)
grid = np.linspace(X[b].quantile(0.05), X[b].quantile(0.95), 20)  # the PDP grid for feature b
synthetic = pd.concat([rows.assign(**{b: v}) for v in grid])      # what the PDP evaluates

# A synthetic point is "unrealistic" if it is farther from any real district than
# 99% of real districts are from their nearest neighbour (in standardized units)
mu, sd = X[[a, b]].mean(), X[[a, b]].std()
nn = NearestNeighbors(n_neighbors=2).fit(((X[[a, b]] - mu) / sd).values)
threshold = np.quantile(nn.kneighbors()[0][:, 0], 0.99)
dist = nn.kneighbors(((synthetic[[a, b]] - mu) / sd).values, n_neighbors=1)[0][:, 0]
off = dist > threshold

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X[a], X[b], s=2, c='lightgray', label='real data')
ax.scatter(synthetic[a][~off], synthetic[b][~off], s=6, c='tab:blue', label='PDP point, realistic')
ax.scatter(synthetic[a][off], synthetic[b][off], s=6, c='tab:red', label='PDP point, unrealistic')
ax.set_xlim(X[a].quantile(0.001), X[a].quantile(0.999)); ax.set_ylim(X[b].quantile(0.001), X[b].quantile(0.999))
ax.set_xlabel(a); ax.set_ylabel(b); ax.legend(loc='best')
ax.set_title(f'Points evaluated by the PDP of {b}: {off.mean():.0%} unrealistic')
plt.show()

## 3. SHAP

SHAP gives each feature its **Shapley value** $\phi_j$: its average marginal contribution over all coalitions of features. They add up to the prediction: $f(x) = E[f(X)] + \sum_j \phi_j(x)$. For trees, **TreeSHAP** computes them exactly and fast (here with respect to a background sample of 100 training districts).

In [ ]:
background = shap.sample(X_train, 100, random_state=SEED)
shap_explainer = shap.TreeExplainer(model, data=background)
shap_values = shap_explainer(X_test)

abs_err = np.abs(y_test.values - y_pred)
idx_worst = int(abs_err.argmax())   # test district with the largest error
print(f"Base value E[f(X)] = {float(shap_values.base_values[0]):.2f} | worst prediction: idx_worst = {idx_worst}")

**Waterfall plot**: from the base value (average prediction), each feature pushes the prediction up (red) or down (blue).

In [ ]:
IDX = 0  # change me: any test index, e.g. idx_worst (why does the model fail there?)

print(f"Test instance {IDX}: true value = {y_test.iloc[IDX]:.2f}, predicted = {y_pred[IDX]:.2f} (x $100k)")
display(X_test.iloc[[IDX]])
shap.plots.waterfall(shap_values[IDX])

**Beeswarm plot**: one dot per district and feature; position = SHAP value, colour = feature value.

In [ ]:
shap.plots.beeswarm(shap_values)

**SHAP dependence plot**: SHAP value of a feature against its value, one dot per district: a "data-driven PDP" evaluated only at real districts. Vertical spread at a fixed value means interactions; the colour shows the feature it interacts with most (or the one you choose).

In [ ]:
DEP_FEATURE = 'MedInc'  # change me: any feature, e.g. 'Latitude', 'AveOccup', 'HouseAge'
COLOR_FEATURE = None    # change me: None (automatic choice) or a feature name, e.g. 'Longitude'

color = shap_values if COLOR_FEATURE is None else shap_values[:, COLOR_FEATURE]
shap.plots.scatter(shap_values[:, DEP_FEATURE], color=color, alpha=0.3,
                   xmin=shap_values[:, DEP_FEATURE].percentile(0.5), xmax=shap_values[:, DEP_FEATURE].percentile(99.5))

**Location effect on a map.** SHAP values are additive, so `SHAP(Latitude) + SHAP(Longitude)` is the total effect of *where* the district is. Separately, each coordinate is misleading: the two are strongly correlated and the model splits the location effect between them arbitrarily.

In [ ]:
shap_lat = shap_values[:, 'Latitude'].values
shap_lon = shap_values[:, 'Longitude'].values
location_effect = shap_lat + shap_lon

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, vals, title in zip(axes, [shap_lon, shap_lat, location_effect],
                           ['SHAP(Longitude)', 'SHAP(Latitude)', 'SHAP(Latitude) + SHAP(Longitude)']):
    sc = ax.scatter(X_test['Longitude'], X_test['Latitude'], c=vals, s=6, alpha=0.8,
                    cmap='RdBu_r', norm=TwoSlopeNorm(vcenter=0))
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.set_title(title)
    ax.set_aspect('equal')
    plt.colorbar(sc, ax=ax, label='SHAP value ($100k)', shrink=0.8)
plt.tight_layout()
plt.show()

## 4. Counterfactuals with DiCE

*What is the smallest change that would bring the predicted value into `DESIRED_RANGE`?* First we let DiCE change anything; then we fix the location and restrict some features to realistic ranges.

In [ ]:
dice_data = dice_ml.Data(dataframe=X_train.assign(MedHouseVal=y_train),
                         continuous_features=feature_names, outcome_name='MedHouseVal')
dice_model = dice_ml.Model(model=model, backend='sklearn', model_type='regressor')
dice_explainer = dice_ml.Dice(dice_data, dice_model, method='random')

def show_cfs(cf_result, query):
    # Table with the query and its counterfactuals; '-' marks unchanged features
    cfs = cf_result.cf_examples_list[0].final_cfs_df
    if cfs is None or len(cfs) == 0:
        print("No counterfactuals found!")
        return None
    table = pd.concat([query[feature_names], cfs[feature_names].astype(float)], ignore_index=True)
    table['prediction'] = model.predict(table[feature_names])
    out = table.round(2).astype(object)
    unchanged = np.isclose(table[feature_names].values, table[feature_names].values[0], rtol=1e-4)
    unchanged[0] = False
    out.loc[:, feature_names] = out[feature_names].mask(unchanged, '-')
    out.index = ['query'] + [f'CF {i + 1}' for i in range(len(cfs))]
    return out

In [ ]:
CF_IDX = 5  # instance of the test set to explain
query = X_test.iloc[[CF_IDX]]
print(f"Test instance {CF_IDX}: predicted value = {model.predict(query)[0]:.2f} (x $100k)")
query

In [ ]:
DESIRED_RANGE = [2.5, 3.5]  # change me: target prediction range (x $100k)
TOTAL_CFS = 4

cf_all = dice_explainer.generate_counterfactuals(query, total_CFs=TOTAL_CFS, desired_range=DESIRED_RANGE,
                                                 random_seed=SEED)
show_cfs(cf_all, query)

Some counterfactuals move the district or use absurd values (e.g. 80 rooms per household): valid for the model, but not actionable. With constraints:

In [ ]:
FEATURES_TO_VARY = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'AveOccup']  # change me
PERMITTED_RANGE = {'AveRooms': [3, 8], 'AveOccup': [1, 5]}                      # change me (or None)

cf_constrained = dice_explainer.generate_counterfactuals(
    query, total_CFs=TOTAL_CFS, desired_range=DESIRED_RANGE, random_seed=SEED,
    features_to_vary=FEATURES_TO_VARY, permitted_range=PERMITTED_RANGE)
show_cfs(cf_constrained, query)

## Summary

| Method | Scope | Main caveat |
|---|---|---|
| Built-in importance | Global | Training data, depends on the definition (gain / weight / cover) |
| Permutation importance | Global | Unrealistic inputs with correlated features; use held-out data |
| PDP / ICE | Global / local | PDP hides interactions (look at ICE); extrapolates with correlated features |
| SHAP | Local + global | Which background / which Shapley value matters; correlated features share credit |
| Counterfactuals | Local | Many valid answers; need plausibility and actionability constraints |